# Camera Discovery Harvest Architecture Test Notebook

This Google Colab notebook tests the `camera-discovery harvest-urls` CLI workflow and the harvest handoff into `camera-discovery run`. Harvest mode is extraction-only: it bypasses target resolution, geocoding, validation, trust policy, scope enforcement, LLM review, GeoJSON, map output, `cameras.md`, and review ZIP generation.

The expanded harvester has two parallel extraction lanes: structured camera/feed records from public JSON/API/GeoJSON/ArcGIS-style endpoints, and raw camera/media URL extraction from HTML, JavaScript, network capture, direct seeds, linked endpoints, and escaped/encoded text. Structured records preserve source-provided fields such as coordinates, orientation/direction, timestamps, `inService`, image descriptions, update frequencies, and grouped media assets. These values are source-provided only; they are not validated, trusted, geocoded, or scope-filtered in harvest mode.


Use `--write-intermediate-records` when you need debug/analysis files for the raw block-policy-filtered records, deduped unique records, and media-filtered records before the final `--max-urls` cap. These files can be large, so the flag is opt-in.


In [ ]:
# Colab setup: run from the checked-out repository root.
from pathlib import Path
import os

REPO_ROOT = Path.cwd()
print('Repository root:', REPO_ROOT)
print('pyproject.toml exists:', (REPO_ROOT / 'pyproject.toml').exists())


In [ ]:
# Install the package in editable mode.
%pip install -e .


In [ ]:
# Verify CLI registration.
!camera-discovery --help
!camera-discovery harvest-urls --help


## Harvest all supported media types

This command collects raw direct media/camera URLs across supported media categories and writes plain URL and metadata outputs. It does not validate streams or write inventory artifacts.


In [ ]:
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california \
  --max-urls 10000 \
  --discovery-mode both \
  --enable-browser-capture


## Harvest only HLS / `.m3u8` URLs


In [ ]:
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california-hls \
  --max-urls 10000 \
  --media .m3u8 \
  --write-intermediate-records


## Inspect optional intermediate harvest records

The HLS harvest command above uses `--write-intermediate-records`, which writes optional debug/analysis JSONL files for the harvest reduction pipeline. `raw_media_records.jsonl` contains block-policy-filtered records before deduplication, `unique_media_records.jsonl` contains deduped records before media filtering, and `media_filtered_records.jsonl` contains records matching the requested media filter before the final `--max-urls` cap.


In [ ]:
from pathlib import Path
import json

HLS_HARVEST_DIR = Path('runs/harvest-california-hls')
intermediate_paths = {
    'raw_media_records.jsonl': HLS_HARVEST_DIR / 'raw_media_records.jsonl',
    'unique_media_records.jsonl': HLS_HARVEST_DIR / 'unique_media_records.jsonl',
    'media_filtered_records.jsonl': HLS_HARVEST_DIR / 'media_filtered_records.jsonl',
    'logs/intermediate_records_summary.json': HLS_HARVEST_DIR / 'logs' / 'intermediate_records_summary.json',
}
for label, path in intermediate_paths.items():
    print(f'{label}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}')

summary_path = HLS_HARVEST_DIR / 'harvest_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('intermediate_records_written:', summary.get('intermediate_records_written'))
    print('intermediate_record_counts:', summary.get('intermediate_record_counts'))
    print('intermediate_record_files:', summary.get('intermediate_record_files'))

preview_path = intermediate_paths['media_filtered_records.jsonl']
if preview_path.exists():
    print('First 3 media-filtered records:')
    with preview_path.open(encoding='utf-8') as f:
        for idx, line in enumerate(f):
            if idx >= 3:
                break
            row = json.loads(line)
            print({k: row.get(k) for k in ['url', 'media_type', 'camera_record_id', 'asset_role', 'source_url', 'source_endpoint_url']})


## Harvest a media mix: HLS, images, and generic stream URLs


In [ ]:
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california-media-mix \
  --max-urls 10000 \
  --media hls,image,stream


## Inspect output paths, summary counts, and structured harvest files

The summary includes counts for structured camera records, grouped media assets, discovered endpoints, records with collected coordinates, orientation, `inService`, update frequencies, and date/time metadata. The CSV/JSONL outputs include top-level fields such as `camera_record_id`, `asset_id`, `asset_role`, `lat`, `lon`, `coordinate_source`, `direction`, `bearing`, `heading`, `orientation`, `in_service`, `date`, `time`, `timestamp`, `last_updated`, `last_refresh`, `image_description`, `current_image_update_frequency`, and `reference_image_update_frequency` when those values were present in source metadata.


In [ ]:
from pathlib import Path
import csv
import json

HARVEST_DIR = Path('runs/harvest-california')
paths = {
    'camera_urls.txt': HARVEST_DIR / 'camera_urls.txt',
    'camera_urls.csv': HARVEST_DIR / 'camera_urls.csv',
    'camera_urls.jsonl': HARVEST_DIR / 'camera_urls.jsonl',
    'camera_records.jsonl': HARVEST_DIR / 'camera_records.jsonl',
    'camera_media_assets.jsonl': HARVEST_DIR / 'camera_media_assets.jsonl',
    'discovered_endpoints.jsonl': HARVEST_DIR / 'discovered_endpoints.jsonl',
    'harvest_camera_inventory.jsonl': HARVEST_DIR / 'harvest_camera_inventory.jsonl',
    'harvest_handoff.json': HARVEST_DIR / 'harvest_handoff.json',
    'harvest_summary.json': HARVEST_DIR / 'harvest_summary.json',
    'source_rows.jsonl': HARVEST_DIR / 'source_rows.jsonl',
    'raw_media_records.jsonl': HARVEST_DIR / 'raw_media_records.jsonl',
    'unique_media_records.jsonl': HARVEST_DIR / 'unique_media_records.jsonl',
    'media_filtered_records.jsonl': HARVEST_DIR / 'media_filtered_records.jsonl',
}
for label, path in paths.items():
    print(f'{label}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}')

summary_path = paths['harvest_summary.json']
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    for key in [
        'raw_records', 'unique_urls', 'media_filtered_urls', 'written_urls',
        'structured_camera_records', 'media_assets', 'endpoints_discovered',
        'records_with_coordinates', 'records_with_orientation', 'records_with_in_service',
        'records_with_datetime', 'records_with_update_frequency',
        'records_with_streaming_video', 'records_with_current_image', 'records_with_reference_image',
        'intermediate_records_written',
    ]:
        print(f'{key}:', summary.get(key))
    print('by_media_type:', summary.get('by_media_type'))
    print('intermediate_record_counts:', summary.get('intermediate_record_counts'))
    print('intermediate_record_files:', summary.get('intermediate_record_files'))
    print('outputs:', summary.get('outputs'))

    print('\nSource row provenance:')
    source_rows_summary = summary.get('source_rows') or {}
    for key in [
        'discovery_mode', 'sources_file', 'sources_file_exists', 'sources_file_loaded', 'sources_file_used',
        'directory_requested', 'blind_requested', 'direct_requested',
        'directory_sources_configured', 'directory_sources_enabled', 'blocked_patterns_configured',
        'generated_rows', 'selected_rows_before_budget', 'selected_rows',
        'selected_directory_rows', 'selected_blind_rows', 'selected_direct_rows',
        'blocked_source_rows', 'max_source_rows', 'max_source_rows_applied',
    ]:
        print(f'{key}:', source_rows_summary.get(key))
    print('generated_by_provider:', source_rows_summary.get('generated_by_provider'))
    print('selected_by_provider:', source_rows_summary.get('selected_by_provider'))
    print('selected_by_kind:', source_rows_summary.get('selected_by_kind'))

    # Cross-check source_rows.jsonl directly so it is obvious whether SOURCES.md/directory rows were present.
    source_rows_path = paths['source_rows.jsonl']
    if source_rows_path.exists():
        from collections import Counter
        source_rows = [json.loads(line) for line in source_rows_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        provider_counts = Counter(row.get('source_provider') or 'unknown' for row in source_rows)
        kind_counts = Counter(row.get('source_kind') or 'unknown' for row in source_rows)
        print('source_rows.jsonl provider counts:', dict(provider_counts))
        print('source_rows.jsonl kind counts:', dict(kind_counts))
        print('first directory/SOURCES.md rows:')
        shown = 0
        for row in source_rows:
            if row.get('source_provider') == 'directory':
                print({k: row.get(k) for k in ['source_name', 'source_kind', 'url', 'source_scope_hint']})
                shown += 1
                if shown >= 5:
                    break
        if shown == 0:
            print('No directory rows found in source_rows.jsonl.')

# Preview flattened URL metadata columns.
csv_path = paths['camera_urls.csv']
if csv_path.exists() and csv_path.stat().st_size:
    with csv_path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for idx, row in enumerate(reader):
            if idx >= 5:
                break
            print({k: row.get(k) for k in [
                'media_url', 'media_type', 'asset_role', 'camera_record_id', 'camera_id',
                'lat', 'lon', 'direction', 'bearing', 'heading', 'in_service',
                'timestamp', 'last_updated', 'image_description',
                'current_image_update_frequency', 'reference_image_update_frequency',
                'source_endpoint_url', 'json_record_path'
            ]})


## Print the first harvested URLs


In [ ]:
N = 25
url_path = Path('runs/harvest-california/camera_urls.txt')
if url_path.exists():
    for idx, line in enumerate(url_path.read_text().splitlines()[:N], start=1):
        print(f'{idx:03d}: {line}')
else:
    print('No camera_urls.txt file found yet.')


## Preview structured camera records, media assets, endpoints, and handoff manifest

These files are the structured harvest artifacts. `camera_records.jsonl` preserves full source camera objects; `camera_media_assets.jsonl` groups direct media URLs by camera; `discovered_endpoints.jsonl` catalogs JSON/API/feed endpoints; `harvest_camera_inventory.jsonl` and `harvest_handoff.json` are used to seed the normal run workflow.


In [ ]:
def preview_jsonl(path: Path, limit: int = 3):
    print(f"
--- {path} ---")
    if not path.exists():
        print("missing")
        return
    for idx, line in enumerate(path.read_text(encoding="utf-8").splitlines()[:limit], start=1):
        try:
            obj = json.loads(line)
            print(idx, json.dumps({k: obj.get(k) for k in list(obj)[:12]}, indent=2)[:2000])
        except Exception:
            print(idx, line[:500])

for file_name in [
    "camera_records.jsonl",
    "camera_media_assets.jsonl",
    "discovered_endpoints.jsonl",
    "harvest_camera_inventory.jsonl",
]:
    preview_jsonl(HARVEST_DIR / file_name)

handoff_path = HARVEST_DIR / "harvest_handoff.json"
if handoff_path.exists():
    print("
--- harvest_handoff.json ---")
    print(json.dumps(json.loads(handoff_path.read_text(encoding="utf-8")), indent=2)[:4000])


## Use harvest handoff as normal run input

This command demonstrates the handoff path. Harvest input is source-provided and unvalidated. Normal `camera-discovery run` still performs target resolution, scope handling, validation/trust/output behavior according to its configuration; the harvest handoff only seeds/enriches candidates so the app can avoid rediscovering metadata that the source already exposed.


In [ ]:
!camera-discovery run "California traffic cameras" \
  --output-dir runs/run-from-harvest \
  --harvest-input runs/harvest-california/harvest_handoff.json


## Optional: zip and download harvest outputs


In [ ]:
from pathlib import Path
import shutil

out_dir = Path('runs/harvest-california')
if out_dir.exists():
    archive = shutil.make_archive(str(out_dir), 'zip', root_dir=out_dir)
    print('Created:', archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:
        print('Download helper unavailable outside Colab:', exc)
else:
    print('Run harvest first; output directory does not exist:', out_dir)
